In [63]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class CatDogDataset(Dataset):

    def __init__(self, cats_dir, dogs_dir, transform=None):

        self.transform = transform
        self.images = []

        for file in os.listdir(cats_dir):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                self.images.append((os.path.join(cats_dir, file), 0))

       
        for file in os.listdir(dogs_dir):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                self.images.append((os.path.join(dogs_dir, file), 1))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        path, label = self.images[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label



dataset = CatDogDataset(
    cats_dir="animals/cats_set",
    dogs_dir="animals/dogs_set",
    transform=transform
)

print("Total Images:", len(dataset))


train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))

images, labels = next(iter(train_loader))

print("Batch Shape :", images.shape)
print("Labels      :", labels)

Total Images: 1000
Train Images: 800
Test Images: 200
Batch Shape : torch.Size([32, 3, 224, 224])
Labels      : tensor([1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 0, 1])


In [64]:
import torch
import torch.nn as nn

class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CatDogCNN()

print(model)

CatDogCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=50176, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [65]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [66]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print("-"*40)

Epoch 1/20
Loss: 20.4146
Train Accuracy: 50.88%
----------------------------------------
Epoch 2/20
Loss: 17.1527
Train Accuracy: 56.38%
----------------------------------------
Epoch 3/20
Loss: 16.2972
Train Accuracy: 63.25%
----------------------------------------
Epoch 4/20
Loss: 15.9507
Train Accuracy: 63.12%
----------------------------------------
Epoch 5/20
Loss: 15.3368
Train Accuracy: 65.50%
----------------------------------------
Epoch 6/20
Loss: 14.7827
Train Accuracy: 67.38%
----------------------------------------
Epoch 7/20
Loss: 13.8333
Train Accuracy: 70.88%
----------------------------------------
Epoch 8/20
Loss: 13.7914
Train Accuracy: 71.12%
----------------------------------------
Epoch 9/20
Loss: 11.9391
Train Accuracy: 77.88%
----------------------------------------
Epoch 10/20
Loss: 11.3448
Train Accuracy: 77.38%
----------------------------------------
Epoch 11/20
Loss: 10.8714
Train Accuracy: 79.50%
----------------------------------------
Epoch 12/20
Loss: 1

In [67]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 66.00%


In [68]:
torch.save(model.state_dict(), "cat_dog_model.pth")

In [69]:
from PIL import Image
from torchvision import transforms
import torch


image = Image.open("cat.4025.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

image = transform(image)
image = image.unsqueeze(0)
image = image.to(device)

In [70]:
model.eval()

with torch.no_grad():
    output = model(image)
    prediction = output.argmax(dim=1).item()

classes = ["Cat", "Dog"]

print("Prediction:", classes[prediction])

Prediction: Cat


In [71]:
from PIL import Image
from torchvision import transforms
import torch


image = Image.open("dog.4015.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

image = transform(image)
image = image.unsqueeze(0)
image = image.to(device)

model.eval()

with torch.no_grad():
    output = model(image)
    prediction = output.argmax(dim=1).item()

classes = ["Cat", "Dog"]

print("Prediction:", classes[prediction])

Prediction: Dog


In [72]:
from PIL import Image
from torchvision import transforms
import torch


image = Image.open("dog.png").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

image = transform(image)
image = image.unsqueeze(0)
image = image.to(device)

model.eval()

with torch.no_grad():
    output = model(image)
    prediction = output.argmax(dim=1).item()

classes = ["Cat", "Dog"]

print("Prediction:", classes[prediction])

Prediction: Dog
